In [1]:
# -----------------------------------------------------------------------------
# STEP 1: INITIALIZE PROJECT ENVIRONMENT
# -----------------------------------------------------------------------------
import os

# Define the local directory structure for data organization
# 'raw' stores original data from APIs; 'processed' will store cleaned data later.
data_directories = ['data/raw', 'data/processed']

for directory in data_directories:
    if not os.path.exists(directory):
        os.makedirs(directory)
        print(f"Directory created: {directory}")
    else:
        print(f"Directory verified: {directory}")

print("Local project environment is ready for data ingestion.")

Directory verified: data/raw
Directory verified: data/processed
Local project environment is ready for data ingestion.


In [10]:
# -----------------------------------------------------------------------------
# STEP 2: DATA ACQUISITION (RATE LIMIT AWARE)
# -----------------------------------------------------------------------------
import pandas as pd
import numpy as np
import requests
import time
from io import StringIO
from datetime import datetime, timedelta, timezone

# Configuration
NASA_URL = "https://firms.modaps.eosdis.nasa.gov/data/active_fire/suomi-npp-viirs-c2/csv/SUOMI_VIIRS_C2_South_Asia_24h.csv"
WEATHER_URL = "https://api.open-meteo.com/v1/forecast"

# --- RATE LIMIT SETTINGS ---
BATCH_SIZE = 100       # Safe batch size
NORMAL_DELAY = 1.5    # Wait 1.5 seconds between every successful request
ERROR_DELAY = 20      # Wait 20 seconds if we hit a 429 error
MAX_RETRIES = 5       # Try 5 times before giving up on a batch
# ---------------------------

def fetch_fire_data():
    print("Fetching satellite data...")
    try:
        response = requests.get(NASA_URL, timeout=30)
        response.raise_for_status()
        
        df = pd.read_csv(StringIO(response.text))
        
        # Spatial Filter (India)
        india_filter = (df['latitude'] >= 8) & (df['latitude'] <= 37) & \
                       (df['longitude'] >= 68) & (df['longitude'] <= 97)
        df_india = df[india_filter].copy()
        
        # Time Filter (Last 24h)
        df_india['acq_time'] = df_india['acq_time'].astype(str).str.zfill(4)
        df_india['dt_utc'] = pd.to_datetime(
            df_india['acq_date'] + ' ' + df_india['acq_time'], 
            format='%Y-%m-%d %H%M'
        )
        # Using 24 hours to ensure we get data
        threshold = datetime.now(timezone.utc).replace(tzinfo=None) - timedelta(hours=24)
        return df_india[df_india['dt_utc'] >= threshold].copy()
    except Exception as e:
        print(f"Error fetching fire data: {e}")
        return None

# EXECUTION
fires_df = fetch_fire_data()

if fires_df is not None:
    # Prepare Data
    fires_df = fires_df[['latitude', 'longitude', 'acq_date', 'frp']].copy()
    fires_df['fire_detected'] = 1
    
    # Generate Safe Zones
    sample_size = max(len(fires_df), 50)
    safe_df = pd.DataFrame({
        'latitude': np.random.uniform(8.0, 37.0, sample_size),
        'longitude': np.random.uniform(68.0, 97.0, sample_size),
        'acq_date': [datetime.now(timezone.utc).strftime('%Y-%m-%d')] * sample_size,
        'frp': 0.0,
        'fire_detected': 0
    })
    
    master_df = pd.concat([fires_df, safe_df], ignore_index=True)
    print(f"Total Target: {len(master_df)} coordinates.")
    print(f"Estimated time: {len(master_df)/BATCH_SIZE * 1.5 / 60:.1f} minutes.")

    # SMART WEATHER LOOP
    weather_data = []
    
    for i in range(0, len(master_df), BATCH_SIZE):
        batch = master_df.iloc[i : i + BATCH_SIZE]
        params = {
            "latitude": ",".join(batch['latitude'].astype(str)),
            "longitude": ",".join(batch['longitude'].astype(str)),
            "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,soil_moisture_0_to_7cm"
        }
        
        # Retry logic for this specific batch
        batch_success = False
        for attempt in range(MAX_RETRIES):
            try:
                r = requests.get(WEATHER_URL, params=params, timeout=30)
                
                # If we hit the rate limit (429), raise an error manually to trigger the except block
                if r.status_code == 429:
                    raise ValueError("Rate Limit Hit")
                
                r.raise_for_status()
                data = r.json()
                
                if isinstance(data, list):
                    weather_data.extend([x.get('current', {}) for x in data])
                else:
                    weather_data.append(data.get('current', {}))
                
                # If we get here, it worked!
                batch_success = True
                time.sleep(NORMAL_DELAY) # Be polite
                break 
                
            except Exception as e:
                # If it was a rate limit, wait long. If it was connection, wait short.
                wait_time = ERROR_DELAY if "Rate Limit" in str(e) else 2
                print(f"  > Batch {i} paused: {e}. Waiting {wait_time}s...")
                time.sleep(wait_time)
        
        if not batch_success:
            print(f"  > Skipping batch {i} after {MAX_RETRIES} failures.")
            weather_data.extend([{} for _ in range(len(batch))])

        if (i + BATCH_SIZE) % 250 == 0:
            print(f"Progress: {min(i + BATCH_SIZE, len(master_df))} / {len(master_df)}")

    # Final Save
    weather_df = pd.DataFrame(weather_data)
    final = pd.concat([master_df, weather_df], axis=1).dropna(subset=['temperature_2m'])
    
    output_path = 'data/raw/master_dataset_real.csv'
    final.to_csv(output_path, index=False)
    print(f"Done! Saved {len(final)} rows to {output_path}")

Fetching satellite data...
Total Target: 4976 coordinates.
Estimated time: 1.2 minutes.
Progress: 500 / 4976
  > Batch 700 paused: Rate Limit Hit. Waiting 20s...
Progress: 1000 / 4976
Progress: 1500 / 4976
Progress: 2000 / 4976
Progress: 2500 / 4976
  > Batch 2800 paused: Rate Limit Hit. Waiting 20s...
Progress: 3000 / 4976
Progress: 3500 / 4976
Progress: 4000 / 4976
  > Batch 4400 paused: Rate Limit Hit. Waiting 20s...
  > Batch 4400 paused: Rate Limit Hit. Waiting 20s...
Progress: 4500 / 4976
Progress: 4976 / 4976
Done! Saved 4976 rows to data/raw/master_dataset_real.csv


In [11]:
# -----------------------------------------------------------------------------
# STEP 3: GEOSPATIAL VALIDATION (DIAGNOSTIC MODE)
# -----------------------------------------------------------------------------
import folium
from global_land_mask import globe
import pandas as pd

# Load the master dataset
try:
    df_raw = pd.read_csv('data/raw/master_dataset_real.csv')
    
    # --- DIAGNOSTIC PRINT ---
    print("--- DATA DIAGNOSTICS ---")
    print(f"Total Rows: {len(df_raw)}")
    fire_count = len(df_raw[df_raw['fire_detected'] == 1])
    safe_count = len(df_raw[df_raw['fire_detected'] == 0])
    print(f"Active Fires Found: {fire_count} (Should be > 0)")
    print(f"Safe Zones Found:   {safe_count}")
    print("------------------------")

    # Filter: Ensure coordinates are on landmass
    df_raw['is_land'] = globe.is_land(df_raw['latitude'], df_raw['longitude'])
    df_final = df_raw[df_raw['is_land']].copy()
    
    print(f"Geospatial Validation: {len(df_final)} points confirmed on land.")
    
    # Initialize interactive map
    # We center it on Central India
    validation_map = folium.Map(location=[20.59, 78.96], zoom_start=5)
    
    # 1. Plot Safe Zones (Green) first so they are at the bottom
    safe_points = df_final[df_final['fire_detected'] == 0]
    for _, row in safe_points.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=4,
            color='green',
            fill=True,
            fill_opacity=0.4,
            popup=f"SAFE: {row.get('temperature_2m')}C"
        ).add_to(validation_map)

    # 2. Plot Fire Zones (Red) second so they appear ON TOP
    fire_points = df_final[df_final['fire_detected'] == 1]
    for _, row in fire_points.iterrows():
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=6, # Slightly larger to stand out
            color='red',
            fill=True,
            fill_color='red',
            fill_opacity=0.8,
            popup=f"FIRE (FRP: {row.get('frp')})"
        ).add_to(validation_map)
    
    # Save validation report
    output_path = 'data/raw/geospatial_validation_report.html'
    validation_map.save(output_path)
    print(f"Map generated: {output_path}")

    # Warning if no fires exist
    if fire_count == 0:
        print("\n⚠️ WARNING: No fires detected in the dataset.")
        print("SOLUTION: Go back to Step 2 and change 'hours=12' to 'hours=24' or 'hours=48'.")

except FileNotFoundError:
    print("Error: Master dataset file not found. Ensure Step 2 executed successfully.")

--- DATA DIAGNOSTICS ---
Total Rows: 4976
Active Fires Found: 2488 (Should be > 0)
Safe Zones Found:   2488
------------------------
Geospatial Validation: 4148 points confirmed on land.
Map generated: data/raw/geospatial_validation_report.html
